---
# `Structure Based Text Splitting in Langchain`
---

- Technique is the most used : this doesn't break sentences or words mid way 
- This ensures that chunks are sematically meaningful and don't break important context

Chunk_size: that we should break word by word, sentence by sentence or paragraph by paragraph
- eg. Text: My Name is Arun (15 char), I live in Delhi (15 char), I am learning GenAI (char 16)

# `Detailed Notes`

# Structured-Based Text Splitting in LangChain

> **Structured-based text splitting divides a document according to its natural structure, such as headings, sections, paragraphs, HTML elements, Markdown headings, or code blocks, instead of blindly splitting after a fixed number of characters.**

The main idea is:

```text
Don't just ask:
"How many characters should each chunk contain?"

Also ask:
"Where does the document naturally change its meaning?"
```

---

# 1. Why Do We Need Structured Splitting?

Suppose you have a technical document:

```text
# Machine Learning

## Supervised Learning

Supervised learning uses labeled data...

## Unsupervised Learning

Unsupervised learning works with unlabeled data...

## Reinforcement Learning

Reinforcement learning learns through rewards...
```

A simple length-based splitter might produce:

```text
Chunk 1:
Supervised learning uses labeled data...
## Unsupervised Lear...

Chunk 2:
ning works with unlabeled data...
```

The heading and its explanation can become separated.

A structured splitter tries to preserve the document's logical organization.

```text
Document
   ↓
Identify Structure
   ↓
Headings / Sections / Paragraphs
   ↓
Meaningful Chunks
```

---

# 2. What Does "Structure" Mean?

Structure depends on the type of document.

### Markdown

```text
# Introduction

## Installation

## Configuration

## Usage
```

Structure:

```text
Heading
   ↓
Subheading
   ↓
Paragraphs
   ↓
Lists
```

### HTML

```html
<h1>Introduction</h1>
<p>...</p>

<h2>Installation</h2>
<p>...</p>
```

### Code

```python
class User:
    ...

def login():
    ...
```

Structure:

```text
Class
Function
Method
```

### Programming language

Structure could be:

```text
Class
 ├── Method
 ├── Method
 └── Method
```

---

# 3. Main Idea

Structured splitting tries to maintain:

> **Semantic and logical boundaries.**

Instead of:

```text
Characters → Characters → Characters
```

we want:

```text
Section → Section → Section
```

or:

```text
Heading + Content
Heading + Content
```

---

# 4. Structured Splitting vs Length-Based Splitting

This is an important distinction.

| Length-Based            | Structure-Based                 |
| ----------------------- | ------------------------------- |
| Focuses on size         | Focuses on document structure   |
| Characters/tokens       | Headings/sections/elements      |
| Easy to implement       | More structure-aware            |
| Can break logical units | Tries to preserve logical units |
| Useful for simple text  | Useful for structured documents |

### Example

Length-based:

```text
Chunk 1 → 1000 characters
Chunk 2 → 1000 characters
Chunk 3 → 1000 characters
```

Structured:

```text
Chunk 1 → Introduction
Chunk 2 → Installation
Chunk 3 → Configuration
Chunk 4 → Usage
```

---

# 5. Structured Text Splitting in LangChain

LangChain provides different splitters for different document structures.

Common examples include:

```text
MarkdownHeaderTextSplitter
HTMLHeaderTextSplitter
RecursiveCharacterTextSplitter
Language-aware code splitters
```

The exact classes and import paths can vary across LangChain versions/integrations, so always check the version of LangChain you're using.

---

# 6. Markdown Header Text Splitter

One of the most useful examples is:

```python
MarkdownHeaderTextSplitter
```

It is designed for Markdown documents.

Suppose you have:

```markdown
# LangChain

LangChain is a framework for building LLM applications.

## Models

Models are used to interact with language models.

## Prompts

Prompts provide instructions to the model.

## Retrievers

Retrievers find relevant documents.
```

The splitter can use headings to understand the document structure.

---

# 7. Basic Example

```python
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = """
# LangChain

LangChain is a framework for building LLM applications.

## Models

Models are used to interact with language models.

## Prompts

Prompts provide instructions to the model.

## Retrievers

Retrievers find relevant documents.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

documents = splitter.split_text(markdown_text)

for doc in documents:
    print(doc.page_content)
    print(doc.metadata)
```

---

# 8. What Happens Internally?

The Markdown:

```text
# LangChain

Introduction...

## Models

Models...

## Prompts

Prompts...

## Retrievers

Retrievers...
```

can become logically separated documents such as:

```text
Document 1
──────────────
Content:
LangChain is a framework...

Metadata:
Header 1 = LangChain
```

```text
Document 2
──────────────
Content:
Models are used...

Metadata:
Header 1 = LangChain
Header 2 = Models
```

```text
Document 3
──────────────
Content:
Prompts provide...

Metadata:
Header 1 = LangChain
Header 2 = Prompts
```

The **metadata is extremely useful**.

---

# 9. Why Metadata Matters

Suppose the retrieved chunk says:

```text
Models are used to interact with language models.
```

Without metadata:

```text
What section did this come from?
Unknown.
```

With metadata:

```text
Header 1: LangChain
Header 2: Models
```

Now you know:

```text
LangChain → Models
```

This can improve:

* Retrieval
* Filtering
* Source attribution
* Debugging
* Context understanding

---

# 10. Important Concept: Hierarchical Structure

Structured documents often have hierarchy:

```text
# LangChain
    │
    ├── ## Models
    │
    ├── ## Prompts
    │
    └── ## Retrievers
```

A deeper structure might be:

```text
# LangChain
    │
    ├── ## Models
    │     ├── ### Chat Models
    │     └── ### LLMs
    │
    ├── ## Prompts
    │     ├── ### Templates
    │     └── ### Few Shot
    │
    └── ## Retrieval
```

Structured splitting can preserve this hierarchy through metadata.

---

# 11. Why Is This Better for RAG?

Imagine a user asks:

> "How do I configure memory in LangChain?"

A good chunk should contain:

```text
LangChain
  ↓
Memory
  ↓
Configuration
  ↓
Relevant explanation
```

rather than an arbitrary 500-character section that might contain:

```text
...end of agents...
...beginning of memory...
...part of callbacks...
```

Structured chunks are often more semantically meaningful.

---

# 12. Structured Splitting + RAG

A typical pipeline:

```text
                 Markdown Documentation
                          ↓
                Structured Splitter
                          ↓
                  Logical Documents
                          ↓
                    Embeddings
                          ↓
                   Vector Database
                          ↓
                     Retriever
                          ↓
                  Relevant Sections
                          ↓
                         LLM
                          ↓
                       Answer
```

---

# 13. Example: Documentation RAG

Suppose you're building:

> **Chat with LangChain Documentation**

Your documentation contains:

```text
# LangChain

## Models

## Prompts

## Chains

## Agents

## Retrievers

## Memory
```

You can split based on these headings.

Then:

```text
Question:
"What is an Agent?"
```

Retriever may find:

```text
Header 1: LangChain
Header 2: Agents

Agents allow an LLM to decide which tools...
```

The retrieved context is much more meaningful.

---

# 14. HTML Structured Splitting

The same idea can be applied to HTML.

Example:

```html
<h1>LangChain</h1>

<h2>Models</h2>
<p>Models are...</p>

<h2>Agents</h2>
<p>Agents are...</p>
```

A header-aware HTML splitter can preserve:

```text
LangChain
  ├── Models
  └── Agents
```

instead of treating the entire HTML page as one giant block.

---

# 15. Why HTML Structure Matters

Web pages contain:

```html
<nav>...</nav>
<header>...</header>
<main>
    <h1>...</h1>
    <h2>...</h2>
    <p>...</p>
</main>
<footer>...</footer>
```

A good ingestion pipeline should ideally focus on meaningful content.

For documentation:

```text
<h1>API Documentation</h1>
<h2>Authentication</h2>
<p>...</p>
```

is much more useful than:

```text
Navigation
Menu
Login
Footer
Cookie message
...
```

So structured extraction and cleaning often work together.

---

# 16. Code-Based Structured Splitting

Code is another excellent use case.

Suppose:

```python
class UserService:

    def create_user(self):
        ...

    def delete_user(self):
        ...


class AuthService:

    def login(self):
        ...
```

A character splitter might cut:

```python
def create_user(self):
    ...
```

in half.

A code-aware splitter tries to preserve logical units such as:

```text
Class
 ↓
Methods
```

or:

```text
Function
 ↓
Function body
```

---

# 17. Language-Aware Splitting

LangChain provides language-aware approaches for programming languages.

Conceptually:

```text
Python
JavaScript
Java
C++
Go
Rust
...
```

The splitter can use language-specific separators or constructs.

For example, Python has:

```text
class
def
```

JavaScript has:

```text
class
function
```

The goal is:

> **Keep code structures together as much as possible.**

---

# 18. Structured Splitting Does NOT Mean "No Chunk Size"

This is an important misconception.

Suppose a section is:

```text
50,000 characters
```

You probably don't want:

```text
One giant chunk
```

just because it belongs to one section.

A practical strategy is often:

```text
First:
Preserve structure

Then:
Control chunk size
```

For example:

```text
Markdown
   ↓
Split by headings
   ↓
Large sections
   ↓
Further split large sections
   ↓
Final chunks
```

This combines **structure + size control**.

---

# 19. Hybrid Strategy

This is often a very good RAG strategy.

```text
Document
   ↓
Structure-Aware Split
   ↓
Sections
   ↓
Check chunk size
   ↓
Large?
 ┌───┴───┐
Yes      No
 ↓        ↓
Further   Keep
split     section
```

For example:

```text
# RAG

## Retrieval
   ↓
500 characters
   ↓
Keep as one chunk

## Advanced Retrieval
   ↓
10,000 characters
   ↓
Further split
```

---

# 20. Structured Splitting vs Recursive Splitting

These concepts are easy to confuse.

### Structured splitting

Uses known document structure:

```text
Markdown headings
HTML headings
Code structure
```

### Recursive character splitting

Uses a hierarchy of separators:

```text
Paragraph
   ↓
Line
   ↓
Sentence/space
   ↓
Character
```

### Simple Mental Model

```text
Structured
→ "What does this document mean structurally?"

Recursive
→ "How can I split this text while keeping it reasonably natural?"
```

---

# 21. When Should You Use Structured Splitting?

Use it when your documents have clear structure.

### Excellent use cases

```text
✓ Markdown documentation
✓ HTML documentation
✓ Technical documentation
✓ API documentation
✓ GitHub README files
✓ Legal documents with sections
✓ Books with chapters
✓ Research papers
✓ Source code
```

---

# 22. When Should You Avoid It?

If the data has little or no meaningful structure:

```text
Random text
Logs
Plain text stream
Chat messages
```

a simple character or recursive splitter may be more appropriate.

---

# 23. Practical GenAI Project

## Project: Chat With Technical Documentation

Imagine you download:

```text
langchain.md
```

Contents:

```text
# LangChain

## Models

...

## Prompts

...

## Chains

...

## Agents

...

## Retrievers

...
```

### Pipeline

```text
langchain.md
      ↓
MarkdownHeaderTextSplitter
      ↓
Sections
      ↓
Further size-based splitting if needed
      ↓
Embeddings
      ↓
Vector Database
      ↓
Retriever
      ↓
LLM
```

---

# 24. Project Code

### Step 1 — Load Markdown

```python
from langchain_community.document_loaders import TextLoader

loader = TextLoader("langchain.md")

documents = loader.load()
```

### Step 2 — Structured Split

```python
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

chunks = splitter.split_text(
    documents[0].page_content
)
```

### Step 3 — Inspect

```python
for chunk in chunks:
    print("CONTENT:")
    print(chunk.page_content)

    print("METADATA:")
    print(chunk.metadata)

    print("-" * 50)
```

You may get:

```text
CONTENT:
Models are used to interact with language models.

METADATA:
{
    'Header 1': 'LangChain',
    'Header 2': 'Models'
}
```

---

# 25. Add Size-Based Splitting

For very large sections:

```text
Markdown
   ↓
Header Splitter
   ↓
Sections
   ↓
RecursiveCharacterTextSplitter
   ↓
Final Chunks
```

Conceptually:

```python
# First: preserve structure
structured_chunks = markdown_splitter.split_text(text)

# Then: control size if required
final_chunks = recursive_splitter.split_documents(
    structured_chunks
)
```

This is a powerful pattern for production RAG.

---

# 26. Important Interview Questions

## Beginner

### Q1. What is structured-based text splitting?

**Answer:**

Structured-based text splitting divides documents according to their natural structure, such as headings, sections, HTML elements, or code constructs, instead of only using fixed character or token lengths.

---

### Q2. Why use structured splitting?

**Answer:**

It preserves logical relationships within documents and produces more semantically meaningful chunks, which can improve retrieval quality in RAG systems.

---

### Q3. Give examples of structured documents.

**Answer:**

Markdown files, HTML pages, technical documentation, API documentation, books, research papers, and source code.

---

# 27. Intermediate Interview Questions

### Q4. What is `MarkdownHeaderTextSplitter`?

**Answer:**

It is a LangChain splitter designed to split Markdown content based on specified Markdown headers and preserve header information in metadata.

---

### Q5. Why is metadata useful when using structured splitting?

**Answer:**

Metadata preserves the document hierarchy, such as chapter, section, and subsection names. It can help with retrieval, filtering, debugging, and source attribution.

---

### Q6. Can a structured splitter replace a size-based splitter completely?

**Answer:**

Not always. A section can still be extremely large. A common production strategy is to first preserve document structure and then apply size-based splitting to oversized sections.

---

# 28. Scenario-Based Interview Questions

### Q7. You have 100 pages of Markdown documentation. Which splitter would you consider?

**Answer:**

I'd consider `MarkdownHeaderTextSplitter` because the documentation likely contains meaningful heading hierarchy. If some sections are very large, I'd combine it with a size-aware splitter.

---

### Q8. Your RAG system retrieves a section but loses its heading/context. What would you do?

**Answer:**

I'd preserve the document hierarchy as metadata during ingestion. For example:

```text
Header 1 → LangChain
Header 2 → Agents
```

This provides additional context for the retrieved chunk.

---

### Q9. Would you use a character splitter for source code?

**Answer:**

I wouldn't make it my first choice. I'd prefer a language-aware/code-aware splitting strategy that tries to preserve functions, classes, and other logical code structures.

---

# 29. 30-Second Revision

> **Structured-based splitting divides text according to its natural document structure rather than blindly using character counts.**

Remember:

```text
Markdown → Headers
HTML     → Headers / Elements
Code     → Classes / Functions
Books    → Chapters / Sections
```

### Main Benefit

```text
Structure
   ↓
Meaningful chunks
   ↓
Better context
   ↓
Potentially better RAG retrieval
```

---

# 30. 2-Minute Revision

## Structured-Based Text Splitting

### Definition

Splits documents according to their logical structure.

### Examples

```text
Markdown
# → ## → ###

HTML
<h1> → <h2> → <h3>

Code
Class → Function → Method
```

### Example

```python
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

chunks = splitter.split_text(markdown_text)
```

### RAG Pipeline

```text
Structured Document
       ↓
Structure-Aware Splitter
       ↓
Logical Sections
       ↓
Size-Based Split if Needed
       ↓
Embeddings
       ↓
Vector DB
       ↓
Retriever
       ↓
LLM
```

### Key Difference

```text
Length-Based
→ "Split based on size."

Structured-Based
→ "Split based on meaning/structure."
```

### Interview One-Liner

> **Structured-based text splitting preserves the natural hierarchy of a document—such as Markdown headings, HTML sections, or code constructs—to create semantically meaningful chunks. In production RAG, it is often combined with size-based splitting so that logical structure is preserved without creating oversized chunks.**
